<a href="https://colab.research.google.com/github/heisenberg304/heart-disease-ml-classification/blob/main/RandomForest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

#===================================================================================================================================================================================================

#MODEL PERFORMANCE

# -----confusion matrix--------
# |                           |
# |          Pred 0   Pred 1  |
# | Real 0     56       28    |
# | Real 1      6       94    |
# -----------------------------
#accuracy  -> 0.82
#precision -> 0.77
#recall    -> 0.94
#f1 score  -> 0.85
#Analisis:
#El modelo logra un recall alto (0.94), lo cual indica que detecta correctamente la mayoría de pacientes con enfermedad.
#Sin embargo, la precisión es menor (0.77), lo que implica falsos positivos.
#Este comportamiento es aceptable dado que en un contexto médico es preferible no dejar pasar casos enfermos.

#Los 3 Importance mas relevantes
#"asymptomatic" [0.22129586] El modelo detecta que la ausencia de síntomas típicos no implica ausencia de enfermedad, lo cual es clínicamente coherente.
#"oldpeak"      [0.12007416] Es una variable médica directa relacionada con problemas cardíacos, por lo que tiene fuerte poder predictivo.
#"exang"        [0.11460541] Indica que el corazón no responde bien al esfuerzo, lo que es un fuerte indicador clínico





In [29]:
#Modelo 2: Separacion de numero de enfermedades
#Cargar librerias
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report


#Cargar dataset
data = pd.read_csv("/content/drive/MyDrive/data_csv/heart_disease_uci.csv")
#===================================================================================================================================================================================================

data = data[data["num"] > 0]

#MANIPULAR CSV
  #rellenar: trestbps(59)(median o mean), chol(30)(median), fbs(90)(mode), restecg(2)(mode), thalch(55)(median o mean), oldpeak(62)(median)
data["trestbps"].fillna(data["trestbps"].median(), inplace=True)
data["chol"].fillna(data["chol"].median(), inplace=True)
data["fbs"].fillna(data["fbs"].mode()[0], inplace=True)
data["restecg"].fillna(data["restecg"].mode()[0], inplace=True)
data["thalch"].fillna(data["thalch"].median(), inplace=True)
data["oldpeak"].fillna(data["oldpeak"].median(), inplace=True)

  #encoding: sex, cp, restecg, num (target)
fe_ma = pd.get_dummies(data["sex"])
CP = pd.get_dummies(data["cp"])
RESTECG = pd.get_dummies(data["restecg"])

  #union de datasets
data = pd.concat([data, fe_ma, CP, RESTECG], axis=1)
  #borrar columnas
data = data.drop(["id", "dataset", "slope", "ca", "thal", "sex", "cp", "restecg"], axis=1)

#===================================================================================================================================================================================================

#crear x and y
x = data[['age','trestbps','chol','fbs','thalch','exang',
          'oldpeak','Female','Male','asymptomatic',
          'atypical angina','non-anginal','typical angina',
          'lv hypertrophy','normal','st-t abnormality']]
y = data["num"]

#split 80/20
x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size = 0.2,
    random_state=55
)

#===================================================================================================================================================================================================

#crear modelo
model = RandomForestClassifier(class_weight = "balanced",
                               n_estimators=200,
                               max_depth=9,
                               min_samples_leaf=6,
                               random_state=1,
                               max_features="sqrt")
#entrenar modelo
model.fit(x_train, y_train)
#prediccion
pred = model.predict(x_test)

#Importancia de cada feature
importances = model.feature_importances_

#===================================================================================================================================================================================================

#metricas
matrix = confusion_matrix(y_test, pred)
todo = classification_report(y_test, pred)
#print de metricas
print(f"todo -> {todo}")
print(f"matrix -> {matrix}")
print(f"Importancia = {importances}")




/tmp/ipykernel_1677/4105293769.py:18: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data["trestbps"].fillna(data["trestbps"].median(), inplace=True)
/tmp/ipykernel_1677/4105293769.py:19: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, in

todo ->               precision    recall  f1-score   support

           1       0.69      0.72      0.70        46
           2       0.24      0.22      0.23        27
           3       0.47      0.30      0.37        23
           4       0.14      0.33      0.20         6

    accuracy                           0.47       102
   macro avg       0.38      0.39      0.38       102
weighted avg       0.49      0.47      0.47       102

matrix -> [[33  5  4  4]
 [11  6  4  6]
 [ 4 10  7  2]
 [ 0  4  0  2]]
Importancia = [0.18115715 0.12651397 0.15179445 0.01875967 0.13567782 0.04059527
 0.19500717 0.00463777 0.00370127 0.01589107 0.00197355 0.0116512
 0.00166245 0.05123013 0.0404935  0.01925355]


In [31]:
%%writefile data_engineering.py

#Modelo 1: Separacion entre enfermos y no enfermos
#Cargar librerias
import pandas as pd
from sklearn.model_selection import train_test_split


#Cargar dataset
data = pd.read_csv("/content/drive/MyDrive/data_csv/heart_disease_uci.csv")

#===================================================================================================================================================================================================

#MANIPULAR CSV
  #rellenar: trestbps(59)(median o mean), chol(30)(median), fbs(90)(mode), restecg(2)(mode), thalch(55)(median o mean), oldpeak(62)(median)
def handle_missing_values(data):
  data["trestbps"].fillna(data["trestbps"].median(), inplace=True)
  data["chol"].fillna(data["chol"].median(), inplace=True)
  data["fbs"].fillna(data["fbs"].mode()[0], inplace=True)
  data["restecg"].fillna(data["restecg"].mode()[0], inplace=True)
  data["thalch"].fillna(data["thalch"].median(), inplace=True)
  data["oldpeak"].fillna(data["oldpeak"].median(), inplace=True)
  return data

data = handle_missing_values(data)

def encode_categorical_features(data):
    #encoding: sex, cp, restecg, num (target)
  fe_ma = pd.get_dummies(data["sex"])
  CP = pd.get_dummies(data["cp"])
  RESTECG = pd.get_dummies(data["restecg"])
  return fe_ma, CP, RESTECG

fe_ma, CP, RESTECG = encode_categorical_features(data)


def feature_selection(data, fe_ma, CP, RESTECG):
    #union de datasets
  data = pd.concat([data, fe_ma, CP, RESTECG], axis=1)
    #borrar columnas
  data = data.drop(["id", "dataset", "slope", "ca", "thal", "sex", "cp", "restecg"], axis=1)
  return data

data = feature_selection(data, fe_ma, CP, RESTECG)
#===================================================================================================================================================================================================

FEATURES = ['age','trestbps','chol','fbs','thalch','exang',
            'oldpeak','Female','Male','asymptomatic',
            'atypical angina','non-anginal','typical angina',
            'lv hypertrophy','normal','st-t abnormality']

##############################################################

#modelo 1
def prepare_binary_dataset(data):
  #copiar dataframe
  data = data.copy()
      #target binario - separar enfermos y no enfermos
  data["num"] = data["num"].apply(lambda x: 1 if x > 0 else 0)
  # crear x , y
  x = data[FEATURES]
  y = data["num"]
  return x, y

x_1, y_1 = prepare_binary_dataset(data)

##############################################################

#modelo 2
def prepare_multiclass_dataset(data):
  #copiar dataframe
  data = data.copy()
      #target multivariable - clasificación de severidad
  data = data[data["num"] > 0]
  # crear x , y
  x = data[FEATURES]
  y = data["num"]
  return x, y

x_2, y_2 = prepare_multiclass_dataset(data)

##############################################################

def split_dataset(x, y):
  #split 80/20
  x_train, x_test, y_train, y_test = train_test_split(
      x,
      y,
      test_size = 0.2,
      random_state=55
  )
  return x_train, x_test, y_train, y_test


Overwriting data_engineering.py


In [33]:
#funciones de data_engineering
from data_engineering import split_dataset

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

x_train, x_test, y_train, y_test = split_dataset(x_1, y_1)

def modelo_1():
  #crear modelo
  model = RandomForestClassifier(class_weight = "balanced",
                                n_estimators=200,
                                max_depth=7,
                                min_samples_leaf=6,
                                random_state=7,
                                max_features="sqrt")
  return model

model = modelo_1()

def entrenar_modelo_1(model, x_train, y_train):
  #entrenar modelo
  model.fit(x_train, y_train)
  return model

model = entrenar_modelo_1(model, x_train, y_train)

def predict_modelo_1(model, x_test):
  #prediccion
  probs = model.predict_proba(x_test)[:, 1]
  threshold = 0.34
  pred = (probs >= threshold).astype(int)
  return pred

pred = predict_modelo_1(model, x_test)

def importances_modelo_1(model):
  #Importancia de cada feature
  importances = model.feature_importances_
  return importances

importances = importances_modelo_1(model)

#===================================================================================================================================================================================================

#metricas
matrix = confusion_matrix(y_test, pred)
todo = classification_report(y_test, pred)

#print de metricas
print(f"matrix -> {matrix}")
print(f"todo -> {todo}")
print(f"Importancia = {importances}")




matrix -> [[56 28]
 [ 7 93]]
todo ->               precision    recall  f1-score   support

           0       0.89      0.67      0.76        84
           1       0.77      0.93      0.84       100

    accuracy                           0.81       184
   macro avg       0.83      0.80      0.80       184
weighted avg       0.82      0.81      0.81       184

Importancia = [0.09069204 0.03292157 0.108115   0.00452154 0.08280342 0.12520588
 0.11291552 0.03655633 0.05179215 0.23815668 0.07238848 0.02198119
 0.00431267 0.00486964 0.00732453 0.00544336]


In [35]:
#funciones de data_engineering
from data_engineering import split_dataset, prepare_multiclass_dataset

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

x_train, x_test, y_train, y_test = split_dataset(x_2, y_2)


def modelo_2():
  #crear modelo
  model = RandomForestClassifier(class_weight = "balanced",
                                n_estimators=200,
                                max_depth=9,
                                min_samples_leaf=6,
                                random_state=1,
                                max_features="sqrt")
  return model

model = modelo_2()

def entrenar_modelo_2(model, x_train, y_train):
  #entrenar modelo
  model.fit(x_train, y_train)
  return model

model = entrenar_modelo_1(model, x_train, y_train)

def predict_modelo_2(model, x_test):
  #prediccion
  pred = model.predict(x_test)
  return pred

pred = predict_modelo_2(model, x_test)

def importances_modelo_2(model):
  #Importancia de cada feature
  importances = model.feature_importances_
  return importances

importances = importances_modelo_2(model)

#===================================================================================================================================================================================================

#metricas
matrix = confusion_matrix(y_test, pred)
todo = classification_report(y_test, pred)

#print de metricas
print(f"matrix -> {matrix}")
print(f"todo -> {todo}")
print(f"Importancia = {importances}")


NameError: name 'x_2' is not defined